# Notification Testing

Test all notification triggers in the Brain System.

### Notification Types
| Type | Trigger | Service |
|------|---------|--------|
| `CONTENT_RELEASED` | Teacher releases content to class | content-workflow-service |
| `ASSIGNMENT_CREATED` | Teacher creates an assignment | content-workflow-service |
| `RECALL_REMINDER` | (not yet automated) | recall-service |
| `ASSIGNMENT_DUE` | (not yet automated) | recall-service |
| `SYSTEM_ANNOUNCEMENT` | Manual/admin broadcast | recall-service |

### Services
| Service | Port |
|---------|------|
| auth-service | 8081 |
| tenant-service | 8082 |
| user-profile-service | 8083 |
| api-gateway | 8084 |
| content-workflow-service | 8086 |
| notes-service | 8088 |
| recall-service | 8091 |

## Setup & Helpers

In [1]:
import requests
import json
import uuid

# --- Configure your server IP here ---
HOST = "http://192.168.0.105"  # Change to your server IP
GW = f"{HOST}:8084"           # API Gateway

# Direct service URLs (bypass gateway)
AUTH_URL     = f"{HOST}:8081"
TENANT_URL   = f"{HOST}:8082"
PROFILE_URL  = f"{HOST}:8083"
WORKFLOW_URL = f"{HOST}:8086"
NOTES_URL    = f"{HOST}:8088"
RECALL_URL   = f"{HOST}:8091"

# --- State ---
ctx = {
    "tenant_id": None,
    "teacher": {},    # {userId, token}
    "student": {},    # {userId, token}
    "class_id": None,
    "note_id": None,
}

# --- Helpers ---
def h(token=None, tenant_id=None, user_id=None):
    """Build request headers."""
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    if tenant_id:
        headers["X-Tenant-Id"] = str(tenant_id)
    if user_id:
        headers["X-User-Id"] = str(user_id)
    return headers

def show(resp, label):
    """Print response nicely."""
    status = resp.status_code
    icon = "OK" if status < 400 else "ERR"
    try:
        body = resp.json()
    except Exception:
        body = resp.text[:300]
    print(f"[{icon}] {label} -> {status}")
    print(json.dumps(body, indent=2) if isinstance(body, (dict, list)) else body)
    print()
    return body if status < 400 else None

print("Helpers loaded.")

/Users/Vinayak/Learner-microservices/flutter-app/scripts/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Helpers loaded.


## Phase 1: Login as Teacher & Student

Login to get tokens. Update email/password to match your seeded users.

In [2]:
# --- Login as Teacher ---
TEACHER_EMAIL = "admin@greenwood.edu"   # Change to your teacher/admin email
TEACHER_PASS  = "Admin@123"

resp = requests.post(f"{GW}/v1/auth/login", json={
    "identifier": TEACHER_EMAIL,
    "password": TEACHER_PASS
})
data = show(resp, "Teacher login")
if data:
    ctx["teacher"] = {
        "userId": data["userId"],
        "token": data["accessToken"],
    }
    ctx["tenant_id"] = data["tenantId"]
    print(f"Tenant ID: {ctx['tenant_id']}")
    print(f"Teacher ID: {ctx['teacher']['userId']}")

[OK] Teacher login -> 200
{
  "accessToken": "b5d2f6bc-0773-4a34-b8a5-f55a79e8d0b9",
  "displayName": "admin@greenwood.edu",
  "expiresIn": 3600,
  "refreshToken": "5fd7fe2c-e3ff-4255-9de8-144666736d91",
  "tenantId": "0b8ea382-2f8b-4de4-9b0e-4747dd3f3c01",
  "userId": "f17eaebd-f023-48fa-9139-e56cd1774ff5",
  "userType": "TEACHER"
}

Tenant ID: 0b8ea382-2f8b-4de4-9b0e-4747dd3f3c01
Teacher ID: f17eaebd-f023-48fa-9139-e56cd1774ff5


In [ ]:
# --- Login as Student ---
STUDENT_EMAIL = "student1@greenwood.edu"  # Change to your student email
STUDENT_PASS  = "Admin@123"

resp = requests.post(f"{GW}/v1/auth/login", json={
    "identifier": STUDENT_EMAIL,
    "password": STUDENT_PASS
})
data = show(resp, "Student login")
if data:
    ctx["student"] = {
        "userId": data["userId"],
        "token": data["accessToken"],
    }
    print(f"Student ID: {ctx['student']['userId']}")

[OK] Student login -> 200
{
  "accessToken": "0c4d801e-e340-42f4-aa55-ff835a99973b",
  "displayName": "student1@greenwood.edu",
  "expiresIn": 3600,
  "refreshToken": "bc80af0b-3034-4ccc-bdde-f7442bfce4dc",
  "tenantId": "2fc10e3c-209e-4acf-927d-1ca26773efe6",
  "userId": "d6f8f817-3f6f-4a6a-bfab-a3101693f534",
  "userType": "TEACHER"
}

Student ID: d6f8f817-3f6f-4a6a-bfab-a3101693f534


## Phase 2: Find Existing Data

Get classes and notes that already exist for this tenant.

In [4]:
# --- List classes for the tenant ---
tid = ctx["tenant_id"]
token = ctx["teacher"]["token"]
uid = ctx["teacher"]["userId"]

resp = requests.get(f"{GW}/v1/tenants/{tid}/classes",
                    headers=h(token, tid, uid))
classes = show(resp, "List classes")
if classes and len(classes) > 0:
    ctx["class_id"] = classes[0]["id"]
    print(f"Using class: {classes[0].get('name', '')} -> {ctx['class_id']}")
else:
    print("No classes found! Create a class first.")

[ERR] List classes -> 404
{
  "timestamp": "2026-02-10T19:21:15.930Z",
  "status": 404,
  "error": "Not Found",
  "path": "/v1/tenants/0b8ea382-2f8b-4de4-9b0e-4747dd3f3c01/classes"
}

No classes found! Create a class first.


In [5]:
# --- List notes for the tenant ---
resp = requests.get(f"{GW}/v1/tenants/{tid}/notes",
                    headers=h(token, tid, uid))
notes = show(resp, "List notes")
if notes and len(notes) > 0:
    ctx["note_id"] = notes[0]["id"]
    ctx["note_title"] = notes[0].get("title", "Untitled")
    print(f"Using note: {ctx['note_title']} -> {ctx['note_id']}")
else:
    print("No notes found! Create a note first.")

[ERR] List notes -> 404
{
  "timestamp": "2026-02-10T19:24:03.712Z",
  "status": 404,
  "error": "Not Found",
  "path": "/v1/tenants/0b8ea382-2f8b-4de4-9b0e-4747dd3f3c01/notes"
}

No notes found! Create a note first.


## Phase 3: Check Current Notifications (Before)

Check what notifications the student currently has.

In [6]:
# --- Get unread count for student ---
student_id = ctx["student"]["userId"]
student_token = ctx["student"]["token"]

resp = requests.get(f"{GW}/v1/notifications/unread/count",
                    headers=h(student_token, tid, student_id))
show(resp, "Student unread count (before)")

[ERR] Student unread count (before) -> 502
Gateway error: finishConnect(..) failed with error(-111): Connection refused: localhost/127.0.0.1:8091



In [7]:
# --- Get all notifications for student ---
resp = requests.get(f"{GW}/v1/notifications?page=0&size=10",
                    headers=h(student_token, tid, student_id))
show(resp, "Student notifications (before)")

[ERR] Student notifications (before) -> 502
Gateway error: finishConnect(..) failed with error(-111): Connection refused: localhost/127.0.0.1:8091



---
## Test 1: Direct Notification (SYSTEM_ANNOUNCEMENT)

Post directly to recall-service notification endpoint. Quickest way to test.

In [8]:
# --- Send a direct notification to the student ---
resp = requests.post(f"{GW}/v1/notifications",
    headers=h(token, tid, uid),
    json={
        "userIds": [student_id],
        "type": "SYSTEM_ANNOUNCEMENT",
        "title": "Welcome to Brain System!",
        "message": "This is a test system announcement. You can safely ignore this.",
        "referenceId": "",
        "referenceType": ""
    }
)
show(resp, "Direct notification (SYSTEM_ANNOUNCEMENT)")

[ERR] Direct notification (SYSTEM_ANNOUNCEMENT) -> 502
Gateway error: finishConnect(..) failed with error(-111): Connection refused: localhost/127.0.0.1:8091



In [ ]:
# --- Verify: check unread count increased ---
resp = requests.get(f"{GW}/v1/notifications/unread/count",
                    headers=h(student_token, tid, student_id))
show(resp, "Student unread count (after direct notification)")

---
## Test 2: Content Release Notification (CONTENT_RELEASED)

Release a note to a class. This triggers `CONTENT_RELEASED` notification to all students in that class.

In [ ]:
# --- Release content to a class ---
# This calls content-workflow-service which notifies students via recall-service
note_id = ctx.get("note_id")
class_id = ctx.get("class_id")
note_title = ctx.get("note_title", "Test Note")

if not note_id or not class_id:
    print("Skipping: need both a note_id and class_id. Check Phase 2 output.")
else:
    resp = requests.post(f"{GW}/v1/tenants/{tid}/releases",
        headers=h(token, tid, uid),
        json={
            "contentIds": [note_id],
            "contentType": "note",
            "classIds": [class_id],
            "contentTitle": note_title
        }
    )
    show(resp, "Release content (triggers CONTENT_RELEASED notification)")

In [ ]:
# --- Verify: check student unread notifications ---
resp = requests.get(f"{GW}/v1/notifications/unread",
                    headers=h(student_token, tid, student_id))
show(resp, "Student unread notifications (after content release)")

---
## Test 3: Assignment Notification (ASSIGNMENT_CREATED)

Create an assignment targeting a class. This triggers `ASSIGNMENT_CREATED` notification.

In [ ]:
# --- Create an assignment ---
class_id = ctx.get("class_id")

if not class_id:
    print("Skipping: need a class_id. Check Phase 2 output.")
else:
    resp = requests.post(f"{GW}/v1/tenants/{tid}/assignments",
        headers=h(token, tid, uid),
        json={
            "title": "Test Assignment - Notification Check",
            "description": "This assignment was created to test notifications.",
            "classIds": [class_id],
            "dueDate": "2026-03-01T23:59:59Z"
        }
    )
    show(resp, "Create assignment (triggers ASSIGNMENT_CREATED notification)")

In [ ]:
# --- Verify: check student unread count ---
resp = requests.get(f"{GW}/v1/notifications/unread/count",
                    headers=h(student_token, tid, student_id))
show(resp, "Student unread count (after assignment)")

---
## Test 4: Bulk Notifications

Send notifications to multiple users at once.

In [ ]:
# --- Get all students in the tenant ---
resp = requests.get(f"{GW}/v1/tenants/{tid}/profiles",
                    headers=h(token, tid, uid))
profiles = show(resp, "List all profiles")

student_ids = []
if profiles:
    for p in profiles:
        if p.get("userType") == "STUDENT":
            student_ids.append(p["userId"])
    print(f"Found {len(student_ids)} students: {student_ids}")

In [ ]:
# --- Send bulk notification to all students ---
if not student_ids:
    print("No students found!")
else:
    resp = requests.post(f"{GW}/v1/notifications",
        headers=h(token, tid, uid),
        json={
            "userIds": student_ids,
            "type": "SYSTEM_ANNOUNCEMENT",
            "title": "School-wide Announcement",
            "message": f"Sent to {len(student_ids)} students as a bulk test.",
            "referenceId": "",
            "referenceType": ""
        }
    )
    show(resp, f"Bulk notification to {len(student_ids)} students")

---
## Notification Management

Read, mark-as-read, and mark-all-as-read.

In [ ]:
# --- Get all student notifications (paginated) ---
resp = requests.get(f"{GW}/v1/notifications?page=0&size=20",
                    headers=h(student_token, tid, student_id))
page = show(resp, "All notifications (page 0)")

# Save first notification ID for mark-as-read test
first_notif_id = None
if page and "content" in page and len(page["content"]) > 0:
    first_notif_id = page["content"][0]["id"]
    print(f"First notification ID: {first_notif_id}")

In [ ]:
# --- Mark a single notification as read ---
if first_notif_id:
    resp = requests.post(f"{GW}/v1/notifications/{first_notif_id}/read",
                         headers=h(student_token, tid, student_id))
    show(resp, f"Mark notification {first_notif_id} as read")
else:
    print("No notification to mark as read.")

In [ ]:
# --- Get unread notifications only ---
resp = requests.get(f"{GW}/v1/notifications/unread",
                    headers=h(student_token, tid, student_id))
show(resp, "Unread notifications")

In [ ]:
# --- Mark ALL notifications as read ---
resp = requests.post(f"{GW}/v1/notifications/read-all",
                     headers=h(student_token, tid, student_id))
show(resp, "Mark all as read")

In [ ]:
# --- Confirm: unread count should be 0 ---
resp = requests.get(f"{GW}/v1/notifications/unread/count",
                    headers=h(student_token, tid, student_id))
show(resp, "Unread count (should be 0)")

---
## Summary

| Test | Endpoint | Notification Type |
|------|----------|------------------|
| Direct notification | `POST /v1/notifications` | `SYSTEM_ANNOUNCEMENT` |
| Content release | `POST /v1/tenants/{tid}/releases` | `CONTENT_RELEASED` |
| Assignment creation | `POST /v1/tenants/{tid}/assignments` | `ASSIGNMENT_CREATED` |
| Bulk notification | `POST /v1/notifications` (multiple userIds) | `SYSTEM_ANNOUNCEMENT` |
| Mark as read | `POST /v1/notifications/{id}/read` | - |
| Mark all as read | `POST /v1/notifications/read-all` | - |
| Get unread count | `GET /v1/notifications/unread/count` | - |
| Get unread list | `GET /v1/notifications/unread` | - |
| Get all (paginated) | `GET /v1/notifications?page=0&size=20` | - |